In [1]:
!pip install tensorflow

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print(tf.__version__)

2.21.0


In [3]:
data = """Cricket is more than just a game in India
It brings millions of people together every single day
I still remember watching the world cup final with my family
We cheered loudly for every run and every wicket
Young players practice for hours in dusty streets and local grounds
They dream of wearing the blue jersey one day
Cricket teaches patience and teamwork to every player
A last over finish can make the calmest heart race
Whether it is a test match or a twenty over game
Cricket will always be my favorite game
The roar of the crowd fills the stadium with energy
Every ball can change the story of the match"""

lines = data.lower().split("\n")
print("Total lines:", len(lines))
print("Total words:", len(data.split()))

Total lines: 12
Total words: 113


In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)
print("Sample word index:", dict(list(tokenizer.word_index.items())[:10]))

Vocabulary size: 81
Sample word index: {'the': 1, 'every': 2, 'a': 3, 'of': 4, 'cricket': 5, 'game': 6, 'and': 7, 'is': 8, 'in': 9, 'it': 10}


In [5]:
input_sequences = []

for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Total sequences:", len(input_sequences))
print("X shape:", X.shape, "| y shape:", y.shape)

Total sequences: 101
X shape: (101, 10) | y shape: (101,)


In [6]:
model = Sequential([
    Embedding(total_words, 64),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, max_len - 1))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 64)         │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100)            │        66,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 81)             │         8,181 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,365 (310.02 KB)

 Trainable params: 79,365 (310.02 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(X, y, epochs=200, batch_size=16, verbose=0)

print("Final loss    :", round(history.history['loss'][-1], 4))
print("Final accuracy:", round(history.history['accuracy'][-1], 4))

Final loss    : 0.0841
Final accuracy: 0.9802


In [8]:
index_word = {idx: word for word, idx in tokenizer.word_index.items()}

def sample_with_temperature(probs, temperature=0.7):
    probs = np.asarray(probs).astype('float64')
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs) / np.sum(np.exp(probs))
    return np.random.choice(len(probs), p=probs)

def generate_text(seed_text, next_words=8, temperature=0.7):
    result = seed_text.lower()
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')
        probs = model.predict(token_list, verbose=0)[0]
        next_index = sample_with_temperature(probs, temperature)
        next_word = index_word.get(next_index, "")
        if next_word == "":
            break
        result += " " + next_word
    return result

seeds = ["cricket", "every", "young"]

print("Generated lines:\n")
for seed in seeds:
    print("-", generate_text(seed, next_words=8))

Generated lines:

- cricket will always be my favorite game india final
- every ball can change the story of the match
- young players practice for hours in dusty streets and


Step	What happens
Tokenizer	Converts each word to an integer to build a vocabulary
Input sequences	Each line is split into growing prefixes so the model learns to predict the next word
Padding	Makes all sequences the same length
Embedding + LSTM	Converts words to vectors, then learns word order and context
Dense (softmax)	Gives a probability for every word in the vocabulary
Generation	Predicts the next word, appends it, and repeats

In [9]:
import os
print(os.listdir())

['.config', 'sample_data']
